In [1]:
import os
import sys

# Add the project root to sys.path
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
# sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
os.chdir("..") # Change working directory to project root

from src.api.core.config import config
from src.api.rag.retrieval import rag_pipeline

import asyncio
from langsmith import Client
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference, LLMContextRecall, NonLLMContextRecall


/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/langsmith/run_helpers.py:480: UserWarning: Unrecognized run_type: reranker. Must be one of: {'tool', 'prompt', 'parser', 'retriever', 'llm', 'embedding', 'chain'}. Did you mean @traceable(name='reranker')?
  warnings.warn(
/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["EVALUATION_MODE"] = "true"

In [3]:
# Initialize LangSmith & Qdrant clients

ls_client = Client(api_key=config.LANGSMITH_API_KEY)

# qdrant_client = QdrantClient(
#     url=f"http://localhost:6333"
# )
qdrant_client = QdrantClient(
    url=config.QDRANT_URL,
    api_key=config.QDRANT_API_KEY  # For Qdrant Cloud only
)

In [ ]:

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


async def ragas_faithfulness(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/
    sample = SingleTurnSample(
            user_input=run.outputs["question"],
            response=run.outputs["answer"],
            retrieved_contexts=run.outputs["retrieved_context"]
        )
    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)


async def ragas_response_relevancy(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_relevance/
    sample = SingleTurnSample(
            user_input=run.outputs["question"],
            response=run.outputs["answer"],
            retrieved_contexts=run.outputs["retrieved_context"]
        )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)


async def ragas_context_precision(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_precision/
    sample = SingleTurnSample(
            user_input=run.outputs["question"],
            response=run.outputs["answer"],
            retrieved_contexts=run.outputs["retrieved_context"]
        )
    scorer = LLMContextPrecisionWithoutReference(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_llm_based(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/
    sample = SingleTurnSample(
            user_input=run.outputs["question"],
            response=run.outputs["answer"],
            reference=example.outputs["ground_truth"],
            retrieved_contexts=run.outputs["retrieved_context"]
        )
    scorer = LLMContextRecall(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_non_llm(run, example):
    # https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/
    sample = SingleTurnSample(
            retrieved_contexts=run.outputs["retrieved_context"],
            reference_contexts=example.outputs["contexts"]
        )
    scorer = NonLLMContextRecall()

    return await scorer.single_turn_ascore(sample)


# results = ls_client.evaluate(
#     lambda x: rag_pipeline(x["question"], qdrant_client, session_id=0),
#     data="rag-evaluation-dataset",
#     evaluators=[
#         ragas_faithfulness,
#         ragas_response_relevancy,
#         ragas_context_precision,
#         ragas_context_recall_llm_based,
#         ragas_context_recall_non_llm
#     ],
#     experiment_prefix="rag-evaluation-dataset"
# )


View the evaluation results for experiment: 'rag-evaluation-dataset-46ac1dfd' at:
https://smith.langchain.com/o/02af5b7b-50d4-4c33-bcba-5ab8f98be641/datasets/e828eebe-7255-4ece-8e57-e4b1faa6de46/compare?selectedSessions=dea2b050-66a9-45e4-8e2b-3cc2e2bf458e




0it [00:00, ?it/s]Error running evaluator <DynamicRunEvaluator ragas_faithfulness> on run a34ec47b-de04-44c3-9745-dfe535c5ce53: RuntimeError('Cannot call `evaluate_run` on an async run evaluator from within an running event loop. Use `aevaluate_run` instead.')
Traceback (most recent call last):
  File "/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1620, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/langsmith/evaluation/evaluator.py", line 340, in evaluate_run
    raise RuntimeError(
RuntimeError: Cannot call `evaluate_run` on an async run evaluator from within an running event loop. Use `aevaluate_run` instead.
Error running evaluator <DynamicRunEvaluator ragas_response_relevancy> on run a34ec47b-de04-44c3-9745-dfe

KeyboardInterrupt: 

In [7]:
results = ls_client.evaluate(
    lambda x: rag_pipeline(x["question"], qdrant_client, session_id=0),
    data="rag-evaluation-dataset",
    evaluators=[
        ragas_faithfulness,
        ragas_response_relevancy,
        ragas_context_precision,
        ragas_context_recall_llm_based,
        ragas_context_recall_non_llm
    ],
    experiment_prefix="rag-evaluation-dataset"
)

# results = await ls_client.aevaluate(
#     target=lambda example: rag_pipeline(
#         example["question"],
#         qdrant_client,
#         session_id=0
#     ),
#     data="rag-evaluation-dataset",
#     evaluators=[
#         ragas_faithfulness,
#         ragas_response_relevancy,
#         ragas_context_precision,
#         ragas_context_recall_llm_based,
#         ragas_context_recall_non_llm,
#     ],
#     experiment_prefix="rag-evaluation-dataset"
# )

# results

# # Async main
# async def main():
#     results = await ls_client.aevaluate(
#         lambda x: rag_pipeline(
#             x["question"],
#             qdrant_client,
#             session_id=0
#         ),
#         data="rag-evaluation-dataset",
#         evaluators=[
#             ragas_faithfulness,
#             ragas_response_relevancy,
#             ragas_context_precision,
#             ragas_context_recall_llm_based,
#             ragas_context_recall_non_llm,
#         ],
#         experiment_prefix="rag-evaluation-dataset"
#     )
#     print(results)

# if __name__ == "__main__":
#     asyncio.run(main())

View the evaluation results for experiment: 'rag-evaluation-dataset-1bfae797' at:
https://smith.langchain.com/o/02af5b7b-50d4-4c33-bcba-5ab8f98be641/datasets/e828eebe-7255-4ece-8e57-e4b1faa6de46/compare?selectedSessions=01df73b6-d325-4dec-b622-55a140fcc3fb




0it [00:00, ?it/s]Error running evaluator <DynamicRunEvaluator ragas_faithfulness> on run 8862e563-d4bc-44d7-a2fd-f6cc613e60dc: RuntimeError('Cannot call `evaluate_run` on an async run evaluator from within an running event loop. Use `aevaluate_run` instead.')
Traceback (most recent call last):
  File "/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1620, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/philippe/Documents/Github/rag-demo/.venv/lib/python3.12/site-packages/langsmith/evaluation/evaluator.py", line 340, in evaluate_run
    raise RuntimeError(
RuntimeError: Cannot call `evaluate_run` on an async run evaluator from within an running event loop. Use `aevaluate_run` instead.
Error running evaluator <DynamicRunEvaluator ragas_response_relevancy> on run 8862e563-d4bc-44d7-a2fd-f6c

KeyboardInterrupt: 

In [6]:
import langsmith
print(langsmith.__version__)

0.4.14
